# Download and convert to Geopackage

This downloads SWOT Pixel Cloud products from hydroweb.next (API-Key necessary) based on a region and a period of interest.
Then is extracts information contained in the area of interest for your study, stores everything in a Geopackage Database for future use.
Geopackage is a convenient data storage format, based on SQL, and is compatible with QGIS.


## Setting the region and period of interest
Using a geopackage layer, preliminary created with, e.g. QGIS, to limit data download and database

In [2]:
from pixcdust.downloaders.hydroweb_next import PixCDownloader
import geopandas as gpd
from datetime import datetime

In [3]:
# reading the area of interest
gdf_geom = gpd.read_file("../data/aoi.gpkg")

# Limiting time period
dates = (
    datetime(2023,4,6),
    datetime(2023,4,8),
)

## Download
This will unfortunately lead to downloading many big files (that will be removed later). This is the only way right now, but the hydroweb.next team is working on improving that.

In [5]:
pixcdownloader = PixCDownloader(
    gdf_geom,
    dates,
    verbose=1,
    path_download='/tmp/pixc1',
    )
pixcdownloader.search_download()

using default backend: py-hydroweb


Wait: 100%|██████████| 100/100 [01:50<00:00,  1.11s/it]
b09cde53-b150-4ae7-b221-c886b1f0b8f7.zip: 100%|██████████| 2.42G/2.42G [00:31<00:00, 76.8MB/s]


In [7]:
!tree /tmp/pixc1

/tmp/pixc1
└── SWOT_L2_HR_PIXC
    ├── SWOT_L2_HR_PIXC_482_016_078L_20230406T094618_20230406T094629_PGC0_01.log
    ├── SWOT_L2_HR_PIXC_482_016_078L_20230406T094618_20230406T094629_PGC0_01.log.md5
    ├── SWOT_L2_HR_PIXC_482_016_078L_20230406T094618_20230406T094629_PGC0_01.met.json
    ├── SWOT_L2_HR_PIXC_482_016_078L_20230406T094618_20230406T094629_PGC0_01.met.json.md5
    ├── SWOT_L2_HR_PIXC_482_016_078L_20230406T094618_20230406T094629_PGC0_01.nc
    ├── SWOT_L2_HR_PIXC_482_016_078L_20230406T094618_20230406T094629_PGC0_01.nc.iso.xml
    ├── SWOT_L2_HR_PIXC_482_016_078L_20230406T094618_20230406T094629_PGC0_01.nc.iso.xml.md5
    ├── SWOT_L2_HR_PIXC_482_016_078L_20230406T094618_20230406T094629_PGC0_01.nc.md5
    ├── SWOT_L2_HR_PIXC_482_016_078L_20230406T094618_20230406T094629_PGC0_01.png
    ├── SWOT_L2_HR_PIXC_482_016_078L_20230406T094618_20230406T094629_PGC0_01.png.md5
    ├── SWOT_L2_HR_PIXC_482_016_078L_20230406T094618_20230406T094629_PGC0_01.rc.xml
    ├── SWOT_L2_HR_PIXC_482_016_0

## Extraction
Now we have all necessary files, let us extract key variables within area of interest in a geopackage database.
This geopackage format is quite efficient (though not the most efficient), and may easily be visualized in, e.g., QGIS
We are using the same geodataframe to limit the data to the area of interest

In [9]:
from pixcdust.converters.gpkg import Nc2GpkgConverter
from glob import glob

In [15]:
# You can specify conditions on variables to filter data
conditions= {"sig0":{'operator': "gt", 'threshold': 20},  # sig0 > 20
             "classification":{'operator': "ge", 'threshold': 3},  # classification >= 3
            }

pixc = Nc2GpkgConverter(
            path_in = glob(pixcdownloader.path_download+'/*/*.nc'),
            variables=['height', 'sig0', 'classification'],
            area_of_interest=gdf_geom,
            conditions=conditions
        )
pixc.database_from_nc(path_out="/tmp/pixc_gpkg.gpkg")

 25%|██▌       | 1/4 [00:28<01:25, 28.45s/it]

skipping layer 20230407_483_16_78L                             (already in geopackage /tmp/pixc_gpkg.gpkg)


100%|██████████| 4/4 [00:45<00:00, 11.28s/it]

skipping layer 20230406_482_16_78L                             (already in geopackage /tmp/pixc_gpkg.gpkg)


database has been succesfully created, we can remove the raw files

In [ ]:
# import shutil
# shutil.rmtree('/tmp/pixc')

# Read the database
Previous steps are not necessary

Now we can open this database in a GeoDataFrame, load it in, e.g., QGIS, etc.

In [17]:
from pixcdust.readers.gpkg import GpkgReader

# nb: you may specify 
pixc_read = GpkgReader(
    "/tmp/pixc_gpkg.gpkg"
)
pixc_read.read()
pixc_read.data

100%|██████████| 2/2 [00:00<00:00,  3.73it/s]


<xarray.Dataset> Size: 2MB
Dimensions:         (index: 36986)
Coordinates:
  * index           (index) int64 296kB 0 1 2 3 4 ... 36982 36983 36984 36985
Data variables:
    height          (index) float32 148kB 324.7 284.0 274.1 ... 192.3 185.2
    sig0            (index) float32 148kB 42.92 20.86 39.61 ... 64.86 29.07
    classification  (index) float32 148kB 6.0 3.0 3.0 3.0 ... 6.0 3.0 6.0 3.0
    geoid           (index) float32 148kB 49.44 49.44 49.42 ... 49.37 49.37
    latitude        (index) float64 296kB 43.52 43.52 43.54 ... 43.68 43.68
    longitude       (index) float64 296kB 1.462 1.459 1.459 ... 1.459 1.459
    wse             (index) float32 148kB 275.2 234.5 224.7 ... 142.9 135.9
    geometry        (index) geometry 296kB POINT (1.4616827424621306 43.51782...

## Rasterization into H3 grid

In [18]:
from pixcdust.converters.gpkg import GpkgDGGSProjecter
h3_grid = GpkgDGGSProjecter("/tmp/pixc_gpkg.gpkg", 10, path_out = '../data/h3_gpkg.gpkg', healpix=False) # True for healpix projection
h3_grid.compute_layers()

Layers: 100%|██████████| 2/2 [00:02<00:00,  1.24s/it]


In [19]:
pixc_read = GpkgReader(
    '../data/h3_gpkg.gpkg'
)
pixc_read.read()
pixc_read.data

100%|██████████| 2/2 [00:00<00:00, 74.26it/s]


<xarray.Dataset> Size: 410kB
Dimensions:         (index: 9328)
Coordinates:
  * index           (index) int64 75kB 0 1 2 3 4 5 ... 9323 9324 9325 9326 9327
Data variables:
    h3_10           (index) int64 75kB 622506101129773055 ... 622506153063448575
    height          (index) float32 37kB 221.2 237.3 242.0 ... 191.6 188.4 189.1
    sig0            (index) float32 37kB 25.33 20.98 36.63 ... 23.45 25.1 56.1
    classification  (index) float32 37kB 3.0 6.0 5.25 3.0 ... 3.0 3.0 3.0 3.0
    geoid           (index) float32 37kB 49.39 49.39 49.39 ... 49.34 49.34 49.34
    wse             (index) float32 37kB 171.8 187.9 192.6 ... 142.3 139.1 139.7
    geometry        (index) geometry 75kB POLYGON ((1.5104330837715698 43.611...

### Display on folium map
Though not the most straightforward and efficient compared to QGIS or other python solutions, this would allow you to dispaly your data in leaflet starting from a GeoDataFrame

In [20]:
pixc_read.layers

['20230407_483_16_78L_10__h3', '20230406_482_16_78L_10__h3']

In [21]:
import folium
import branca.colormap as cmp

layer_h3 = pixc_read.read_single_layer('20230407_483_16_78L_10__h3')

# creating a colormap
linear = cmp.LinearColormap(
    ['blue', 'purple', 'orange', 'yellow'],
    vmin=layer_h3['wse'].min(),  # minimum wse value
    vmax=layer_h3['wse'].max(),  # maximum wse value
    caption='Water Surface Elevation (m)' #Caption for Color scale or Legend
)
# Initiating map
m = folium.Map([43.6, 1.43], zoom_start=12, tiles="cartodbpositron")

folium.GeoJson(
    layer_h3,
    style_function = lambda row:  {
        'fillColor': linear(row['properties']["wse"]),
        'weight': 0,          #how thick the border has to be
        'fillOpacity': 1
    },
).add_to(m)
linear.add_to(m)   #adds colorscale and legend
m

Enjoy !